# 署名を作成しよう

- データに署名を行うことでデータの改竄を防ぐことができます。

- ハイブリッド暗号の仕組みを見てみよう
---

### ・セクション1: 関数の準備

In [1]:
#ライブラリのインストール
%pip install pycryptodome


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.Hash import SHA256
from Crypto.Signature import pss
import os

In [3]:
#公開鍵の生成
def generate_keys():
    key = RSA.generate(2048)
    private_key = key
    public_key = key.publickey()
    return private_key, public_key

In [4]:
#共通鍵の作成とメッセージの暗号化
def hybrid_encrypt(message, public_key):
    #1.共通鍵（セッションキー）の生成
    session_key = os.urandom(32)
    
    #2.共通鍵を送信相手の公開鍵で暗号化
    cipher_rsa = PKCS1_OAEP.new(public_key, hashAlgo=SHA256)
    encrypted_key = cipher_rsa.encrypt(session_key)
    
    #3.共通鍵でメッセージを暗号化
    cipher_aes = AES.new(session_key, AES.MODE_GCM)
    nonce = cipher_aes.nonce
    
    #4.暗号化と同時に認証タグを生成
    c, tag = cipher_aes.encrypt_and_digest(message)
    
    #暗号化された共通鍵、nonce、署名、暗号文を配列としてして返す
    ans = [encrypted_key, nonce, tag, c]
    return ans

In [5]:
#復号関数
def hybrid_decrypt(encrypted_key, nonce, tag, c, private_key):
    #1.秘密鍵でセッションキーを復号
    cipher_rsa = PKCS1_OAEP.new(private_key, hashAlgo=SHA256)
    session_key = cipher_rsa.decrypt(encrypted_key)
    
    #2.共通鍵でメッセージを復号
    cipher_aes = AES.new(session_key, AES.MODE_GCM, nonce=nonce)
    
    #3.復号と同時に認証タグの検証を行う
    try:
        m = cipher_aes.decrypt_and_verify(ciphertext, tag)
        return m
    except ValueError:
        raise ValueError("メッセージまたはタグが改ざんされています (認証失敗)")

In [6]:
#署名の作成(送信側)
def sign_message(data, private_key):
    h = SHA256.new(data)
    signer = pss.new(private_key)
    signature = signer.sign(h)
    return signature

#署名の検証(受信側)
def verify_signature(data, signature, public_key):
    h = SHA256.new(data)
    verifier = pss.new(public_key)
    
    try:
        verifier.verify(h, signature)
        return True
    except (ValueError, TypeError):
        return False

---

### ・セクション2: 先生にメッセージを送ってみよう

In [9]:
#生徒の暗号化用
S_private_key, S_public_key = generate_keys()
print(S_private_key, S_public_key)

Private RSA key at 0x112281CD0 Public RSA key at 0x112283710


In [10]:
"""
演習課題(1)
"""
#送信処理

#メッセージ作成
message = input("メッセージを入力: ").encode('utf-8')

#デジタル署名
signature = sign_message(message, #鍵の変数名を入れよう)

#ハイブリッド暗号化
ans = hybrid_encrypt(message, #鍵の変数名を入れよう)

                                                               
print(#通信に必要な変数を出力しよう)                                                        

SyntaxError: incomplete input (2211078384.py, line 16)

In [8]:
#先生からのメッセージが届いたか確認してみよう
try:
    decrypted_message = hybrid_decrypt(encrypted_session_key, nonce, tag, ciphertext, S_private_key)
    print("メッセージが復号されました。")
    
    #デジタル署名の検証
    is_valid = verify_signature(decrypted_message, signature, T_public_key)
    print(f"署名の検証結果: {is_valid}")

    if is_valid:
        print(f"メッセージは教師から送信され、改ざんされていないことが確認できました。")
        print(f"復号されたメッセージ: 「{decrypted_message.decode()}」")
    else:
        print("署名が不正です。送信者が異なるか、メッセージが改ざんされています。")
        
except ValueError as e:
    print(f"復号失敗: {e}")

メッセージが復号されました。
署名の検証結果: True
メッセージは教師から送信され、改ざんされていないことが確認できました。
復号されたメッセージ: 「テスト」
